# Phân tích hậu kỳ thực nghiệm

Notebook này chỉ đọc các file CSV kết quả đã có, không huấn luyện lại mô hình. Kết quả được xuất vào `results/posthoc/`.

In [ ]:
from pathlib import Path
import ast
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

# Đường dẫn dataset đã giải nén trên Kaggle.
DATA_ROOT = Path('/kaggle/input/datasets/nguyendinhtanloc1')
AL_DIR = DATA_ROOT / 'al-results' / 'al_results'
ABLATION_DIR = DATA_ROOT / 'ablation-results'

# Kiểm tra đúng hai dataset đầu vào trước khi đọc dữ liệu.
if not AL_DIR.exists():
    raise FileNotFoundError(f'Không tìm thấy dataset AL: {AL_DIR}')
if not ABLATION_DIR.exists():
    raise FileNotFoundError(f'Không tìm thấy dataset ablation: {ABLATION_DIR}')

OUT = Path('/kaggle/working/posthoc_results')
TABLES = OUT / 'tables'
FIGURES = OUT / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print('AL_DIR:', AL_DIR)
print('ABLATION_DIR:', ABLATION_DIR)
print('OUT:', OUT)

In [ ]:
# Đọc file tổng hợp thực nghiệm chính.
# ƯU TIÊN file dạng combined_results_YYYYMMDD_....csv (có đầy đủ cột
# map50_95, batch_diversity). File main_experiment/combined_results.csv
# chỉ có cột entropy đã tách và KHÔNG có map50_95.
all_candidates = sorted(AL_DIR.rglob('combined_results*.csv'))
if not all_candidates:
    raise FileNotFoundError(f'Không tìm thấy combined_results*.csv trong {AL_DIR}')

pref = [p for p in all_candidates if '_seed' not in p.name and p.name != 'combined_results.csv']
main_path = pref[-1] if pref else all_candidates[-1]
print('Các file candidate:', [p.name for p in all_candidates])
main = pd.read_csv(main_path)

# combined_results_*.csv lưu entropy trong batch_diversity dạng chuỗi dict.
if 'batch_entropy' not in main.columns:
    if 'batch_diversity' not in main.columns:
        raise ValueError('Không tìm thấy batch_entropy hoặc batch_diversity')

    def parse_diversity(value):
        obj = ast.literal_eval(value) if isinstance(value, str) else value
        if not isinstance(obj, dict):
            raise ValueError(f'batch_diversity không hợp lệ: {value!r}')
        weather = float(obj.get('weather', np.nan))
        scene = float(obj.get('scene', np.nan))
        timeofday = float(obj.get('timeofday', np.nan))
        return pd.Series({
            'weather_entropy': weather,
            'scene_entropy': scene,
            'timeofday_entropy': timeofday,
            'batch_entropy': weather + scene + timeofday,
        })

    parsed = main['batch_diversity'].apply(parse_diversity)
    for column in parsed.columns:
        main[column] = parsed[column]
    if 'batch_diversity' in main.columns:
        main = main.drop(columns=['batch_diversity'])

# domain_map50.csv nằm trong thư mục main_experiment nếu đã upload.
domain_candidates = sorted(AL_DIR.rglob('domain_map50*.csv'))
domains = pd.read_csv(domain_candidates[-1]) if domain_candidates else None

abl_candidates = sorted(ABLATION_DIR.rglob('abl_*.csv'))
if not abl_candidates:
    raise FileNotFoundError(f'Không tìm thấy abl_*.csv trong {ABLATION_DIR}')
abl_runs = {path.stem: pd.read_csv(path) for path in abl_candidates}

required = {'pipeline', 'seed', 'round', 'labeled_count', 'map50', 'batch_entropy'}
missing = required - set(main.columns)
if missing:
    raise ValueError(f'Missing columns in {main_path.name}: {missing}')

main['pipeline_label'] = main['pipeline'].str.replace('_', ' ').str.title()
main.to_csv(TABLES / 'raw_main_results.csv', index=False)

print('Main file:', main_path)
print('Has map50_95:', 'map50_95' in main.columns)
print('Main shape:', main.shape)
print('Ablation files:', list(abl_runs))
print('Domain file:', domain_candidates[-1].name if domain_candidates else 'not found')
display(main.head())

In [ ]:
summary = (main.groupby(['pipeline','round','labeled_count'])[['map50','batch_entropy']]
          .agg(['mean','std','min','max']).reset_index())
summary.columns = ['_'.join(c).strip('_') if isinstance(c, tuple) else c for c in summary.columns]
summary.to_csv(TABLES / 'main_summary_by_round.csv', index=False)
display(summary)

In [ ]:
def paired_stats(metric, round_no):
    d = main[main['round'].eq(round_no)].pivot(index='seed', columns='pipeline', values=metric).dropna()
    base = d['baseline'].to_numpy(float)
    strat = d['stratified'].to_numpy(float)
    diff = strat - base
    t, p2 = stats.ttest_rel(strat, base)
    p1 = p2 / 2 if t > 0 else 1 - p2 / 2
    sd = diff.std(ddof=1)
    cd = diff.mean() / sd if sd else np.nan
    return {'metric': metric, 'round': round_no, 'n': len(diff),
            'baseline_mean': base.mean(), 'stratified_mean': strat.mean(),
            'mean_diff_strat_minus_base': diff.mean(), 't': t,
            'p_two_sided': p2, 'p_one_sided_strat_gt_base': p1,
            'cohens_d_paired': cd}
tests = pd.DataFrame([paired_stats(m,r) for m in ['batch_entropy','map50'] for r in sorted(main['round'].unique())])
tests.to_csv(TABLES / 'paired_tests.csv', index=False)
display(tests)

In [ ]:
corr = []
for metric in ['map50']:
    x = main['batch_entropy'].to_numpy(float)
    y = main[metric].to_numpy(float)
    r, p = stats.pearsonr(x, y)
    rs, ps = stats.spearmanr(x, y)
    corr.append({'entropy':'batch_entropy','metric':metric,'n':len(x),'pearson_r':r,'pearson_p':p,'spearman_r':rs,'spearman_p':ps})
corr = pd.DataFrame(corr)
corr.to_csv(TABLES / 'entropy_map_correlation.csv', index=False)
display(corr)

In [ ]:
# Tạo bảng ablation từ các CSV riêng lẻ.
# L1 quota không được lưu trong CSV ablation, nên các giá trị dưới đây được
# nhập từ log [QUOTA] của lần chạy ablation.
L1_FROM_LOGS = {
    'equal':   'weather 122 / scene 46 / timeofday 106',
    'inverse': 'weather 276 / scene 40 / timeofday 276',
    'sqrt':    'weather 0 / scene 4 / timeofday 2',
}

def parse_entropy(value):
    if isinstance(value, dict):
        return float(sum(value.values()))
    if isinstance(value, str):
        try:
            obj = ast.literal_eval(value)
            return float(sum(obj.values())) if isinstance(obj, dict) else float(value)
        except Exception:
            return float(value)
    return float(value)

def value_from_row(row, column):
    """Lấy một giá trị từ pandas Series; trả NaN nếu cột không tồn tại."""
    if column not in row.index or pd.isna(row[column]):
        return np.nan
    return float(row[column])

abl_rows = []

# Baseline: lấy từ file tổng hợp đã chọn ở cell đọc dữ liệu.
base_r2 = main[(main.pipeline == 'baseline') & (main.seed == 42) & (main['round'] == 2)].iloc[0]
abl_rows.append({
    'config': 'baseline',
    'quota_mode': 'none',
    'quota_scope': 'none',
    'hybrid_top_k': 0,
    'entropy_r2': float(base_r2['batch_entropy']),
    'overall_map50_r2': float(base_r2['map50']),
    'overall_map50_95_r2': value_from_row(base_r2, 'map50_95'),
    'val_map50_r2': float(base_r2['val_map50']),
    'quota_l1_r2': 'n/a',
})

for name, df in abl_runs.items():
    row = df[df['round'].eq(2)].iloc[0]
    mode = name.replace('abl_', '').replace('_seed42', '')
    abl_rows.append({
        'config': name,
        'quota_mode': mode,
        'quota_scope': 'per_field' if mode in ('equal', 'inverse') else 'joint',
        'hybrid_top_k': 3 if mode in ('equal', 'inverse') else 0,
        'entropy_r2': parse_entropy(row['batch_diversity']),
        'overall_map50_r2': float(row['map50']),
        'overall_map50_95_r2': value_from_row(row, 'map50_95'),
        'val_map50_r2': float(row['val_map50']),
        'quota_l1_r2': L1_FROM_LOGS.get(mode, 'not stored'),
    })

# Sqrt = Stratified seed 42 (main experiment).
sq = main[(main.pipeline == 'stratified') & (main.seed == 42) & (main['round'] == 2)].iloc[0]
abl_rows.append({
    'config': 'stratified_seed42_sqrt',
    'quota_mode': 'sqrt',
    'quota_scope': 'per_field',
    'hybrid_top_k': 3,
    'entropy_r2': float(sq['batch_entropy']),
    'overall_map50_r2': float(sq['map50']),
    'overall_map50_95_r2': value_from_row(sq, 'map50_95'),
    'val_map50_r2': float(sq['val_map50']),
    'quota_l1_r2': L1_FROM_LOGS['sqrt'],
})

ablation = pd.DataFrame(abl_rows)
ablation.to_csv(TABLES / 'ablation_quota_summary.csv', index=False)

if domains is not None:
    domains.to_csv(TABLES / 'domain_map50.csv', index=False)
display(ablation)
if domains is None:
    print('CẢNH BÁO: không tìm thấy domain_map50.csv trong dataset al-results')

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
colors = {'baseline':'#4C78A8','stratified':'#F58518'}
# Entropy theo vòng
fig, ax = plt.subplots(figsize=(8,5))
for pipe, g in main.groupby('pipeline'):
    s = g.groupby('round')['batch_entropy'].agg(['mean','std']).reset_index()
    ax.errorbar(s['round'], s['mean'], yerr=s['std'], marker='o', capsize=4, label=pipe.title(), color=colors[pipe])
ax.set(xlabel='Vòng Active Learning', ylabel='Batch entropy (bit)', title='Entropy theo vòng')
ax.legend(); fig.tight_layout(); fig.savefig(FIGURES/'entropy_by_round.png', dpi=200); plt.show()

# mAP50 theo ngân sách
fig, ax = plt.subplots(figsize=(8,5))
for pipe, g in main.groupby('pipeline'):
    s = g.groupby('labeled_count')['map50'].agg(['mean','std']).reset_index()
    ax.errorbar(s['labeled_count'], s['mean'], yerr=s['std'], marker='o', capsize=4, label=pipe.title(), color=colors[pipe])
ax.set(xlabel='Số ảnh đã chọn', ylabel='mAP50', title='mAP50 theo ngân sách'); ax.legend(); fig.tight_layout(); fig.savefig(FIGURES/'map50_by_budget.png', dpi=200); plt.show()

In [ ]:
# Biểu đồ mAP50 theo miền (chỉ chạy khi dataset có domain_map50.csv).
if domains is None:
    print('Bỏ qua biểu đồ miền: al-results không chứa domain_map50.csv.')
    print('Các phần entropy, mAP theo ngân sách, kiểm định và ablation vẫn được tạo bình thường.')
else:
    fig, ax = plt.subplots(figsize=(9,5))
    x = np.arange(len(domains)); w = .36
    ax.bar(x-w/2, domains['baseline_map50'], w, label='Baseline', color=colors['baseline'])
    ax.bar(x+w/2, domains['stratified_map50'], w, label='Stratified', color=colors['stratified'])
    ax.set_xticks(x, domains['domain'], rotation=20, ha='right')
    ax.set_ylabel('mAP50')
    ax.set_title('mAP50 theo miền dữ liệu')
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES/'domain_map50.png', dpi=200)
    plt.show()

# Ablation entropy và mAP50
# Nếu các file CSV ablation không chứa bảng tổng hợp, bảng đã được tạo ở cell trước.
a = ablation.copy(); x=np.arange(len(a)); w=.36
fig, ax1 = plt.subplots(figsize=(9,5)); ax2=ax1.twinx()
ax1.bar(x-w/2, a['entropy_r2'], w, label='Entropy', color='#72B7B2')
ax2.bar(x+w/2, a['overall_map50_r2'], w, label='mAP50', color='#E45756')
ax1.set_xticks(x, a['config'], rotation=25, ha='right')
ax1.set_ylabel('Entropy (bit)')
ax2.set_ylabel('mAP50')
ax1.set_title('Ablation quota ở vòng 2')
fig.tight_layout()
fig.savefig(FIGURES/'ablation_entropy_map50.png', dpi=200)
plt.show()

# L1 quota; chỉ vẽ các cấu hình có số liệu L1 thực tế.
l1_rows=[]
for _, row in a.iterrows():
    value = row.get('quota_l1_r2', 'n/a')
    if isinstance(value, str) and value not in ('n/a', 'not stored in CSV'):
        vals = [int(v) for v in re.findall(r'\d+', value)]
        if len(vals) >= 3:
            l1_rows.append({'config':row['config'], 'weather':vals[0], 'scene':vals[1], 'timeofday':vals[2]})
l1=pd.DataFrame(l1_rows)
l1.to_csv(TABLES/'quota_l1.csv',index=False)
if l1.empty:
    print('Bỏ qua biểu đồ L1: dataset hiện tại không chứa chi tiết sai số quota.')
else:
    ax=l1.set_index('config').plot.bar(figsize=(8,5))
    ax.set_ylabel('Sai số L1')
    ax.set_title('Sai số quota theo trường')
    plt.tight_layout()
    plt.savefig(FIGURES/'quota_l1.png',dpi=200)
    plt.show()

In [ ]:
# Scatter entropy - mAP50
fig, ax = plt.subplots(figsize=(8,5))
for pipe, g in main.groupby('pipeline'):
    ax.scatter(g['batch_entropy'], g['map50'], label=pipe.title(), color=colors[pipe], s=55)
ax.set(xlabel='Batch entropy (bit)', ylabel='mAP50', title='Quan hệ giữa entropy và mAP50')
ax.legend(); fig.tight_layout(); fig.savefig(FIGURES/'entropy_map50_scatter.png', dpi=200); plt.show()

report = ['POST-HOC STATISTICAL REPORT', '', tests.to_string(index=False), '', corr.to_string(index=False)]
(OUT/'statistical_report.txt').write_text('\n'.join(report), encoding='utf-8')

# Đóng gói toàn bộ kết quả hậu kỳ cần dùng cho báo cáo.
# Không đưa model/checkpoint/ảnh eval vào ZIP này.
import zipfile
ZIP_PATH = Path('/kaggle/working/posthoc_results.zip') if Path('/kaggle/working').exists() else Path('posthoc_results.zip')
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT.rglob('*')):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(OUT).as_posix())

print('Đã xuất bảng vào', TABLES)
print('Đã xuất hình vào', FIGURES)
print('Đã xuất ZIP:', ZIP_PATH)
print('Các file trong ZIP:')
with zipfile.ZipFile(ZIP_PATH) as zf:
    for name in zf.namelist():
        print(' -', name)